# Module 2 · Add one business tool
**Approved function:** `check_loan_status(application_id)`  ·  ⏱ ~15 min

The application will:
1. Understand the customer request
2. **Decide** whether the tool is needed
3. **Call** the function (our code, not the model, runs it)
4. Use the result to generate the final response

In [1]:
import os, sys, json
sys.path.append("..")                      # so we can import utils.py and harness/

from dotenv import load_dotenv
import aisuite as ai
from utils import print_html, print_json, show_messages, show_tool_call, show_trace, show_usage, compare

load_dotenv("../.env", override=True)
MODEL = os.getenv("MODEL", "openai:gpt-4o-mini")   # aisuite format -> "provider:model"
client = ai.Client()                               # reads OPENAI_API_KEY from the environment

print_html({"model": MODEL, "client": "aisuite.Client() ready"}, title="Setup", color="gray")

model,openai:gpt-4o-mini
client,aisuite.Client() ready


## The approved business function
In real life this would call the Loan Origination System (LOS) API. Here it reads `data/loans.json` (synthetic data).

In [10]:
from harness.tools import load_loans
DB = load_loans()   # synthetic data in data/loans.json; dates shifted so they always look current

def check_loan_status(application_id: str) -> dict:
    """Look up the current status of a loan application.

    Args:
        application_id: Loan application ID in the format LA-1234.
    """
    app = DB["applications"].get(application_id.strip().upper())
    if not app:
        return {"found": False, "error": f"No application {application_id}"}
    return {"found": True, "application_id": application_id, "loan_type": app["loan_type"],
            "status": app["status"], "stage": app["stage"],
            "missing_documents": app["missing_documents"],
            "expected_decision_by": app["expected_decision_by"]}

print_html(check_loan_status("LA-1003"), title="Calling the function directly")

found,True
application_id,LA-1003
loan_type,Car Loan
status,APPROVED
stage,Awaiting disbursal
missing_documents,[]
expected_decision_by,None


## Describe the tool to the model (JSON schema)
The model never sees our Python code. It only sees this **contract**: the name, what the tool does, and its parameters.

In [11]:
TOOLS = [{
    "type": "function",
    "function": {
        "name": "check_loan_status",
        "description": "Look up the current status of a loan application. Use whenever the "
                       "customer asks about the status or progress of their application.",
        "parameters": {
            "type": "object",
            "properties": {
                "application_id": {"type": "string", "description": "Loan ID in the format LA-1234"}
            },
            "required": ["application_id"],
        },
    },
}]

SYSTEM_PROMPT = """You are the Loan Service Assistant for Demo Bank.
Use the check_loan_status tool for any question about a specific application. Never guess a status.
Be concise and polite. You cannot approve or reject loans."""
print_json(TOOLS, title="Tool schema sent to the model")

## Step 1 · The model *decides* to use the tool

In [13]:
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "Hi, how much time would it take to process loan ?"},
]
first = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS, temperature=0)
msg = first.choices[0].message

print_html({"content": msg.content, "finish_reason": first.choices[0].finish_reason},
           title="Model response #1: no text, a tool request instead", color="amber")
print_json([tc.model_dump() for tc in msg.tool_calls], title="tool_calls")

content,"The processing time for a loan can vary based on several factors, including the type of loan and the completeness of the application. Typically, it can take anywhere from a few days to several weeks. For specific information regarding your application, please provide your loan application ID, and I can check the status for you."
finish_reason,stop


TypeError: 'NoneType' object is not iterable

## Step 2 · *Our code* executes the approved function

In [5]:
tool_call = msg.tool_calls[0]
args = json.loads(tool_call.function.arguments)
result = check_loan_status(**args)          # <- the application runs it, not the LLM

show_tool_call(tool_call.function.name, args, result)

application_id,LA-1002
found,True
application_id,LA-1002
loan_type,Personal Loan
status,DOCUMENTS_PENDING
stage,Document verification
missing_documents,"[ ""Last 3 months salary slips"", ""Bank statement (last 6 months)"" ]"
expected_decision_by,None


## Step 3 · Send the result back → the model writes the final answer

In [6]:
messages.append({
    "role": "assistant",
    "content": msg.content,
    "tool_calls": [{"id": tool_call.id, "type": "function",
                    "function": {"name": tool_call.function.name,
                                 "arguments": tool_call.function.arguments}}],
})
messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": json.dumps(result)})

final = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS, temperature=0)
show_messages(messages + [final.choices[0].message], title="Full round-trip")

## Put it together: does the model *always* call the tool?
No. The **model decides**. A general question needs no lookup.

In [7]:
def ask(question: str) -> str:
    """One request, at most one tool round-trip."""
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": question}]
    r = client.chat.completions.create(model=MODEL, messages=msgs, tools=TOOLS, temperature=0)
    m = r.choices[0].message
    if not m.tool_calls:
        return m.content, "answered directly"
    msgs.append({"role": "assistant", "content": m.content, "tool_calls": [
        {"id": tc.id, "type": "function",
         "function": {"name": tc.function.name, "arguments": tc.function.arguments}} for tc in m.tool_calls]})
    for tc in m.tool_calls:
        msgs.append({"role": "tool", "tool_call_id": tc.id,
                     "content": json.dumps(check_loan_status(**json.loads(tc.function.arguments)))})
    r2 = client.chat.completions.create(model=MODEL, messages=msgs, tools=TOOLS, temperature=0)
    return r2.choices[0].message.content, f"used tool: {[tc.function.name for tc in m.tool_calls]}"

for q in ["What is the status of LA-1003?", "What is an EMI?"]:
    answer, how = ask(q)
    print_html(answer, title=q, subtitle=how, color="green" if "tool" in how else "blue")

## Before vs after

In [8]:
no_tool = client.chat.completions.create(model=MODEL, temperature=0, messages=[
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "What is the status of LA-1002?"}]).choices[0].message.content
with_tool, _ = ask("What is the status of LA-1002?")
compare(no_tool, with_tool, "LLM only", "LLM + approved tool")

## aisuite shortcut: pass Python functions, let aisuite run the loop
aisuite reads the **docstring and type hints** to build the schema, then runs the call → execute → respond loop for up to `max_turns`.

In [9]:
from aisuite.utils.tools import Tools
print_json(Tools([check_loan_status]).tools(), title="Schema auto-generated from the docstring")

In [10]:
auto = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "system", "content": SYSTEM_PROMPT},
              {"role": "user", "content": "Any update on LA-1001?"}],
    tools=[check_loan_status],     # plain Python function!
    max_turns=3,
)
show_messages(auto.choices[0].intermediate_messages, title="What aisuite did for us (intermediate messages)")
print_html(auto.choices[0].message.content, title="Final answer", color="green")

> ⚠️ **Why do we still write the loop ourselves in BFSI?** The auto-loop is great for prototypes, but it hides the control points we need: an **allow-list** check before execution, **PII masking**, **audit logging** of every call, and **escalation** when something looks wrong. The Module 5 harness writes its own loop for exactly this reason.

## ✅ Takeaways
- A **tool** = a schema the model sees + a function **your code** runs.
- The model **decides** *if* and *which* tool to call. Your application **executes** it.
- Tool results ground the answer in real data → no hallucinated statuses.